# California Housing Price Prediction
### End-to-end regression project (Hands-On Machine Learning, Ch. 2)

A single, linear path from raw data to a saved, tuned model:

1. Load & explore the data
2. Create a stratified train/test split
3. Build a preprocessing pipeline (custom features + scaling + encoding)
4. Compare candidate models
5. Fine-tune the best model
6. Evaluate on the test set
7. Save the final model


## 1. Load the data

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# The dataset: California housing prices, one row per census block group.
url = "https://raw.githubusercontent.com/ageron/handson-ml2/master/datasets/housing/housing.csv"
housing = pd.read_csv(url)

housing.head()


In [ ]:
# Quick structural check: column dtypes and null counts.
# "total_bedrooms" has missing values -> we will impute them in the pipeline.
housing.info()


In [ ]:
# Summary statistics for every numeric column.
housing.describe()


In [ ]:
# Visualize the distribution of every numeric attribute at once.
housing.hist(bins=50, figsize=(20, 15))
plt.show()


## 2. Train / test split (stratified by income)

A plain random split can, by chance, under- or over-represent certain income
brackets in the test set. Median income is one of the strongest predictors of
house value, so we split in a way that preserves its distribution: bucket it
into an `income_cat` column and use `StratifiedShuffleSplit` on that column.

In [ ]:
# Bucket median_income into 5 categories, used only for stratified sampling.
housing["income_cat"] = pd.cut(
    housing["median_income"],
    bins=[0., 1.5, 3.0, 4.5, 6., np.inf],
    labels=[1, 2, 3, 4, 5],
)
housing["income_cat"].hist()
plt.show()


In [ ]:
from sklearn.model_selection import StratifiedShuffleSplit

# 80/20 split, stratified on income_cat so both sets mirror the full
# dataset's income distribution.
splitter = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_index, test_index = next(splitter.split(housing, housing["income_cat"]))

strat_train_set = housing.loc[train_index].reset_index(drop=True)
strat_test_set = housing.loc[test_index].reset_index(drop=True)

# income_cat was only a helper column for the split -> drop it from both sets.
for set_ in (strat_train_set, strat_test_set):
    set_.drop("income_cat", axis=1, inplace=True)

len(strat_train_set), len(strat_test_set)


In [ ]:
# From here on, work only with the training set. Separate predictors from
# the label so the target never leaks into the preprocessing pipeline.
housing = strat_train_set.drop("median_house_value", axis=1)
housing_labels = strat_train_set["median_house_value"].copy()

housing.shape, housing_labels.shape


## 3. Preprocessing pipeline

One consistent pipeline, built with `ColumnTransformer`, that:

- turns raw counts into more informative **ratios** (bedrooms/rooms, rooms/household, people/household)
- **log-transforms** heavily skewed columns (rooms, bedrooms, population, households, income)
- adds a **geographic similarity** feature: how close each district is to cluster centers of similar districts (via k-means + RBF kernel)
- **one-hot encodes** the categorical column (`ocean_proximity`)
- **imputes + scales** every remaining numeric column

Wrapping all of this in one pipeline means the exact same transformations are
applied to training data, test data, and any new data at prediction time.

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.cluster import KMeans
from sklearn.metrics.pairwise import rbf_kernel


class ClusterSimilarity(BaseEstimator, TransformerMixin):
    """Custom transformer: clusters districts by (lat, lon) with k-means,
    then measures each district's similarity (RBF kernel) to every
    cluster center. Produces one "similarity" feature per cluster."""

    def __init__(self, n_clusters=10, gamma=1.0, random_state=None):
        self.n_clusters = n_clusters
        self.gamma = gamma
        self.random_state = random_state

    def fit(self, X, y=None, sample_weight=None):
        self.kmeans_ = KMeans(self.n_clusters, random_state=self.random_state)
        self.kmeans_.fit(X, sample_weight=sample_weight)
        return self

    def transform(self, X):
        return rbf_kernel(X, self.kmeans_.cluster_centers_, gamma=self.gamma)

    def get_feature_names_out(self, names=None):
        return [f"Cluster {i} similarity" for i in range(self.n_clusters)]


In [ ]:
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler


def column_ratio(X):
    """Divide column 0 by column 1, e.g. total_bedrooms / total_rooms."""
    return X[:, [0]] / X[:, [1]]


def ratio_pipeline():
    """Impute -> compute a ratio -> scale. Reused for every ratio feature."""
    return make_pipeline(
        SimpleImputer(strategy="median"),
        FunctionTransformer(column_ratio, feature_names_out=lambda ft, names: ["ratio"]),
        StandardScaler(),
    )


# Impute -> log-transform (compresses long right-tailed distributions) -> scale.
log_pipeline = make_pipeline(
    SimpleImputer(strategy="median"),
    FunctionTransformer(np.log, feature_names_out="one-to-one"),
    StandardScaler(),
)

# Geographic feature: similarity to 10 k-means clusters of (lat, lon).
cluster_simil = ClusterSimilarity(n_clusters=10, gamma=1.0, random_state=42)

# Most frequent value for the categorical column, then one-hot encode it.
cat_pipeline = make_pipeline(
    SimpleImputer(strategy="most_frequent"),
    OneHotEncoder(handle_unknown="ignore"),
)

# Default path for any numeric column not explicitly listed below:
# impute missing values with the median, then standardize.
default_num_pipeline = make_pipeline(
    SimpleImputer(strategy="median"), StandardScaler()
)

preprocessing = ColumnTransformer([
    ("bedrooms_ratio", ratio_pipeline(), ["total_bedrooms", "total_rooms"]),
    ("rooms_per_house", ratio_pipeline(), ["total_rooms", "households"]),
    ("people_per_house", ratio_pipeline(), ["population", "households"]),
    ("log", log_pipeline, ["total_bedrooms", "total_rooms", "population",
                            "households", "median_income"]),
    ("geo", cluster_simil, ["latitude", "longitude"]),
    ("cat", cat_pipeline, make_column_selector(dtype_include=object)),
], remainder=default_num_pipeline)  # applies to housing_median_age, lat, lon are handled above

preprocessing


## 4. Compare candidate models

Fit the preprocessing + model as one pipeline and score each candidate with
10-fold cross-validation on the training set, using RMSE (lower is better).
Cross-validation is used here (instead of a single train/validation split) to
get a more reliable estimate of how each model generalizes.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor


def evaluate(model, name):
    rmses = -cross_val_score(
        model, housing, housing_labels,
        scoring="neg_root_mean_squared_error", cv=10,
    )
    print(f"{name:>18}: mean RMSE = {rmses.mean():,.0f}  (std = {rmses.std():,.0f})")
    return rmses


lin_reg = make_pipeline(preprocessing, LinearRegression())
tree_reg = make_pipeline(preprocessing, DecisionTreeRegressor(random_state=42))
forest_reg = make_pipeline(preprocessing, RandomForestRegressor(random_state=42))

_ = evaluate(lin_reg, "Linear Regression")
_ = evaluate(tree_reg, "Decision Tree")
_ = evaluate(forest_reg, "Random Forest")


Random Forest is expected to come out on top (lowest mean RMSE) — it's
the one we fine-tune next. (Decision Tree tends to badly overfit: very low
training error but a high cross-validation error.)

## 5. Fine-tune the Random Forest

Search over the two hyperparameters that matter most here: the number of
geographic clusters used as a feature, and the number of features considered
at each split of the forest. `RandomizedSearchCV` samples a fixed number of
random combinations, which scales much better than a full grid search.

In [ ]:
from scipy.stats import randint
from sklearn.model_selection import RandomizedSearchCV
from sklearn.pipeline import Pipeline

full_pipeline = Pipeline([
    ("preprocessing", preprocessing),
    ("random_forest", RandomForestRegressor(random_state=42)),
])

param_distribs = {
    "preprocessing__geo__n_clusters": randint(low=3, high=50),
    "random_forest__max_features": randint(low=2, high=20),
}

rnd_search = RandomizedSearchCV(
    full_pipeline,
    param_distributions=param_distribs,
    n_iter=10,
    cv=3,
    scoring="neg_root_mean_squared_error",
    random_state=42,
)
rnd_search.fit(housing, housing_labels)

print("Best params:", rnd_search.best_params_)
print(f"Best CV RMSE: {-rnd_search.best_score_:,.0f}")


In [ ]:
# Which features matter most to the tuned forest? Useful for sanity-checking
# the model and for possibly dropping low-value features later.
final_model = rnd_search.best_estimator_

feature_importances = final_model["random_forest"].feature_importances_
feature_names = final_model["preprocessing"].get_feature_names_out()

importances = sorted(zip(feature_importances, feature_names), reverse=True)
for importance, name in importances[:10]:
    print(f"{importance:.3f}  {name}")


## 6. Evaluate on the test set

The test set has been untouched until now. This is the one number that
estimates how the model will perform on genuinely new data.

In [ ]:
from sklearn.metrics import root_mean_squared_error

X_test = strat_test_set.drop("median_house_value", axis=1)
y_test = strat_test_set["median_house_value"].copy()

final_predictions = final_model.predict(X_test)
final_rmse = root_mean_squared_error(y_test, final_predictions)

print(f"Final RMSE on the test set: {final_rmse:,.0f}")


## 7. Save the final model

In [ ]:
import joblib

joblib.dump(final_model, "my_california_housing_model.pkl")

# Reload check: load it back and confirm it predicts without error.
final_model_reloaded = joblib.load("my_california_housing_model.pkl")
final_model_reloaded.predict(X_test.iloc[:5])
